# RQ1 — Meta-IFD matched: `Meta-IFD` vs `Meta-IFD+` (Bảng A: split paper · Bảng B: 10 split mới)

Notebook này đo **đúng đóng góp của 12 UFE** cho Meta-IFD. Hai biến thể dùng **cùng mã HMSL chính thức**,
**cùng cấu hình mặc định**, **cùng sampled batches** và **cùng một realization ICVAE views** trong mỗi split.
Khác nhau duy nhất: bản `+` nối 12 UFE (không dùng nhãn, `log1p`) vào mọi hàng của mọi view (14 → 26 chiều).

| Bảng | Dữ liệu | Seed | Kiểm định | Vai trò |
|---|---|---|---|---|
| **A** | split paper `1657/553/553` | `[12..16]` | paired node bootstrap | retrospective matched comparison (test đã từng được xem) |
| **B** | 10 split phân tầng mới `[101, 202, …, 1010]` | `[12..16]` mỗi split | Nadeau–Bengio trên **10 chênh lệch cấp split** (trung bình 5 seed trước) | pre-registered re-split stability — **không** phải external holdout |

Split paper **không** nằm trong 10 split của Bảng B. Tổng: 2×5 + 10×2×5 = **110 lần train**.

Quy tắc khóa trước: cấu hình Meta-IFD mặc định (hidden 128, lr 1e-3, λ 0,1, 3 views, patience 50, quy tắc checkpoint
chính thức `>=`), threshold 0,5, checkpoint chọn bằng validation, test mỗi split chỉ được lấy mẫu sau khi đã khóa
checkpoint của **cả hai** biến thể. **Kết luận chính (Bảng B):** UFE cải thiện Meta-IFD khi Δ ≥ 0,5 điểm Macro-F1
**và** cận dưới CI 95% Nadeau–Bengio > 0. Không đổi protocol nếu kết quả không đạt.

Notebook lưu kết quả và **giữ toàn bộ checkpoint** của từng split. Nếu phiên Kaggle bị ngắt, attach output cũ làm input và chạy lại:
một split chỉ được nạp lại khi khớp protocol hash, code hash, data/source hash, hash ID train/val/test, hash `score_cube.npz`,
hash mọi checkpoint, và Macro-F1 tính lại từ cube trùng giá trị đã lưu; nếu không, split đó được train lại.


In [1]:
from pathlib import Path
import csv
import gc
import hashlib
import importlib
import json
import os
import random
import shutil
import subprocess
import sys
import time
from datetime import datetime, timezone

import numpy as np
import torch

EXPECTED_SOURCE_SHA256 = {
    "model.py": "86fcd312eb004500afe72eea6a151189224d9d289f930f34819f974ca3970e3e",
    "run.py": "0cf5488678c04809c4f69c4f6779039b3b85cadea3f952da50a44181f59539f8",
    "attention_conv.py": "0080935c16553a59b498c275c1decab96c5d5569a0194f8435cae6ef4db44c1e",
    "icvae_Phish.pkl": "536c13bc765039c7e1d0fa5ec168e9bfbe47ad92aebd156ccc73de6d194d0bf8",
}
SOURCE_DATA_SHA256 = "8cc653bb389ea1a316f72c3893d0487ab09321552ae022ceaa89cd2499f690e8"
VARIANTS = ["base", "plus"]
VARIANT_LABEL = {"base": "Meta-IFD", "plus": "Meta-IFD+"}
MODEL_SEEDS = [12, 13, 14, 15, 16]
PAPER_SAMPLER_SEEDS = {"train": 1201, "validation": 1202, "test": 2301}
PAPER_VIEW_SEEDS = {"train": 1203, "validation": 1204, "test": 2302}
RESPLIT_SEEDS = [101, 202, 303, 404, 505, 606, 707, 808, 909, 1010]
EXPECTED_COUNTS = {"train": 1657, "validation": 553, "test": 553}
EDGE_STATS_DIM = 12
RELATION_TO_INDEX = {"call": 0, "trans": 1}
BOOTSTRAP_REPETITIONS = 20_000
MINIMUM_PRACTICAL_DELTA = 0.005
ALPHA = 0.05
CONFIG = {
    "dataset": "HEIG-Phish", "target_node": "EOA",
    "hidden": 128, "learning_rate": 0.001, "contrastive_lambda": 0.1, "views": 3,
    "batch_size": 512, "fanout": [100, 100], "weight_decay": 5e-4,
    "max_epochs": 1000, "patience": 50, "threshold": 0.5,
    "checkpoint_rule": "official Meta-IFD run.py: validation Macro-F1 >= best resets patience; stop when stale > patience; best checkpoint restored",
    "tuning": "none; official default configuration for both variants",
    "model_seeds": MODEL_SEEDS,
    "seed_policy": "independent: lazy layers materialised by one no-grad forward, then set_seed(seed) + reset_parameters(); no dropout or explicit random operation during training (fixed batch/view order), but some CUDA/PyG kernels may still be non-deterministic",
    "batch_policy": "per split: train/validation/test batches sampled once and reused identically by both variants and all seeds",
    "view_policy": "per split: one ICVAE realization (3 views x (6 edge-type augmentations + original)) generated once and shared by both variants and all seeds",
    "only_difference_between_variants": "plus concatenates the same 12 label-free UFE statistics (log1p) to every row of every view (14 -> 26 dims)",
    "ufe": "rows/count/sum x in/out x call/trans inside each sampled batch; labels hidden before computation",
    "table_a": {"split": "paper masks 1657/553/553", "sampler_seeds": PAPER_SAMPLER_SEEDS, "view_seeds": PAPER_VIEW_SEEDS,
                "test": "paired node bootstrap of mean per-seed Macro-F1, 20000 reps", "role": "retrospective matched comparison"},
    "table_b": {"split_seeds": RESPLIT_SEEDS, "split_method": "stratified 60/20/20 over the 2763 labeled EOA (sklearn train_test_split twice)",
                "sampler_seeds": "train/val/test = split_seed*10 + 1/2/3", "view_seeds": "train/val/test = split_seed*10 + 4/5/6",
                "unit_of_analysis": "split-level delta = mean over 5 seeds (plus) - mean over 5 seeds (base)",
                "test": "Nadeau-Bengio corrected resampled t-test, k=10, df=9, ratio n_test/n_train = 553/1657",
                "role": "pre-registered re-split stability; not an external holdout; paper split excluded"},
    "decision_rule": "primary (Table B): UFE improves Meta-IFD iff mean delta >= 0.005 and NB 95% CI lower bound > 0",
    "no_change_policy": "protocol is not modified if the rule is not met",
}
CODE_SHA256 = "51c3b9db274e8f16bb896e92bf1be7a48656a40d2f69bf73ba8010c2228535b3"  # SHA-256 of all code cells with this constant replaced by __CODE_SHA256__; written by the generator
CONFIG["code_sha256"] = CODE_SHA256
PROTOCOL_JSON = json.dumps(CONFIG, sort_keys=True, indent=2) + "\n"
PROTOCOL_SHA256 = hashlib.sha256(PROTOCOL_JSON.encode("utf-8")).hexdigest()


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def sha256_int64(values):
    return hashlib.sha256(np.asarray(values, dtype=np.int64).tobytes()).hexdigest()


try:
    import torch_geometric
    import pyg_lib
    import torch_scatter
    import torch_sparse
except ModuleNotFoundError:
    torch_version = torch.__version__.split("+")[0]
    cuda_tag = "cpu" if torch.version.cuda is None else "cu" + torch.version.cuda.replace(".", "")
    wheel_url = f"https://data.pyg.org/whl/torch-{torch_version}+{cuda_tag}.html"
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch_geometric"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "pyg_lib", "torch_scatter", "torch_sparse", "-f", wheel_url])
    import torch_geometric
    import pyg_lib
    import torch_scatter
    import torch_sparse

from torch_geometric.data import HeteroData
from torch_geometric.loader import NeighborLoader

assert torch.cuda.is_available(), "Hãy bật GPU trong Kaggle"
device = torch.device("cuda:0")
timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
run_dir = Path("/kaggle/working") / f"rq1_matched_metaifd_A_B_{timestamp}"
run_dir.mkdir(parents=True, exist_ok=False)
(run_dir / "protocol.json").write_text(PROTOCOL_JSON, encoding="utf-8")
assert sha256_file(run_dir / "protocol.json") == PROTOCOL_SHA256
print("Protocol SHA256:", PROTOCOL_SHA256)

# Đúng snapshot mã công bố đã audit.
source_candidates = [
    p.parent for p in Path("/kaggle/input").rglob("model.py")
    if (p.parent / "run.py").is_file() and (p.parent / "attention_conv.py").is_file()
    and (p.parent / "pretrain_model/icvae_Phish.pkl").is_file()
]
source_candidates = [
    p for p in source_candidates
    if sha256_file(p / "model.py") == EXPECTED_SOURCE_SHA256["model.py"]
    and sha256_file(p / "run.py") == EXPECTED_SOURCE_SHA256["run.py"]
    and sha256_file(p / "attention_conv.py") == EXPECTED_SOURCE_SHA256["attention_conv.py"]  # excludes patched copies from resumed outputs
    and sha256_file(p / "pretrain_model/icvae_Phish.pkl") == EXPECTED_SOURCE_SHA256["icvae_Phish.pkl"]
]
assert len(source_candidates) >= 1, "Không tìm thấy snapshot Meta-IFD-main đã audit"
source_dir = source_candidates[0]
for name, path in {"model.py": source_dir / "model.py", "run.py": source_dir / "run.py",
                   "attention_conv.py": source_dir / "attention_conv.py",
                   "icvae_Phish.pkl": source_dir / "pretrain_model/icvae_Phish.pkl"}.items():
    assert sha256_file(path) == EXPECTED_SOURCE_SHA256[name], name
repo = run_dir / "Meta-IFD-official-copy"
shutil.copytree(source_dir, repo, ignore=shutil.ignore_patterns("data", "__pycache__"))

# Compatibility patch (giống notebook Meta-IFD đã audit): PairTensor hint cho PyG hiện đại.
attention_file = repo / "attention_conv.py"
content = attention_file.read_text(encoding="utf-8")
old_import = "from torch_geometric.typing import EdgeType, Metadata, NodeType, SparseTensor"
new_import = "from torch_geometric.typing import EdgeType, Metadata, NodeType, PairTensor, SparseTensor"
old_propagate = "# propagate_type: (k: Tensor, q: Tensor, v: Tensor, rel: Tensor)"
new_propagate = "# propagate_type: (v: PairTensor)"
assert old_import in content and old_propagate in content
attention_file.write_text(content.replace(old_import, new_import).replace(old_propagate, new_propagate), encoding="utf-8")

os.chdir(repo)
sys.path.insert(0, str(repo))
for module_name in ("model", "attention_conv", "icvae", "rcvae"):
    sys.modules.pop(module_name, None)
importlib.invalidate_caches()
official_model_module = importlib.import_module("model")
HMSL = official_model_module.HMSL
assert Path(official_model_module.__file__).resolve() == (repo / "model.py").resolve()

data_candidates = [p for p in Path("/kaggle/input").rglob("data.pt")
                   if "phish/processed/data.pt" in str(p).replace("\\", "/").lower()]
assert len(data_candidates) == 1, [str(p) for p in data_candidates]
data_sha256 = sha256_file(data_candidates[0])
assert data_sha256 == SOURCE_DATA_SHA256
loaded = torch.load(data_candidates[0], map_location="cpu", weights_only=False)
data = loaded[0] if isinstance(loaded, tuple) and len(loaded) == 2 else loaded
assert isinstance(data, HeteroData)
del loaded
gc.collect()
assert set(data.node_types) == {"EOA", "CA"}
assert data["EOA"].x.shape[1] == data["CA"].x.shape[1] == 14
paper_masks = {"train": data["EOA"].train_mask.clone(), "validation": data["EOA"].val_mask.clone(),
               "test": data["EOA"].test_mask.clone()}
assert tuple(int(m.sum()) for m in paper_masks.values()) == (1657, 553, 553)
labeled_mask = (data["EOA"].y == 0) | (data["EOA"].y == 1)
labeled_ids = torch.where(labeled_mask)[0].numpy()
labeled_y = data["EOA"].y[labeled_mask].numpy()
assert len(labeled_ids) == 2763 and int(labeled_y.sum()) == 1206
fanout_by_relation = {edge_type: CONFIG["fanout"] for edge_type in data.edge_types}
all_edge_types = list(data.edge_types)
edge_type_to_id = {edge_type: idx for idx, edge_type in enumerate(all_edge_types)}
assert len(all_edge_types) == 6

(repo / "rcvae.py").write_text("from icvae import ICVAE, Encoder, Decoder\nRCVAE = ICVAE\n", encoding="utf-8")
sys.modules.pop("rcvae", None)
importlib.invalidate_caches()
import rcvae
icvae_model = torch.load(repo / "pretrain_model/icvae_Phish.pkl", map_location="cpu", weights_only=False).to(device)
icvae_model.eval()

# Resume: nạp kết quả split đã xong từ output phiên trước (nếu được attach làm input) với cùng protocol hash.
previous_split_results = {}
for path in Path("/kaggle/input").rglob("split_result.json"):
    try:
        item = json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        continue
    if item.get("protocol_sha256") == PROTOCOL_SHA256 and item.get("code_sha256") == CODE_SHA256:
        previous_split_results[item["split_key"]] = (item, path.parent)  # full integrity check in reuse_or_run
print("Resumable splits found:", sorted(previous_split_results))
print("Torch/PyG:", torch.__version__, torch_geometric.__version__, "| GPU:", torch.cuda.get_device_name(0))
print("Output:", run_dir)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 20.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 36.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 76.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 83.0 MB/s eta 0:00:00
Protocol SHA256: 29a9e98c57a2984aea0d84a2b3671b1da1c3a0a9783ee5082891820744b9262e
Resumable splits found: []
Torch/PyG: 2.10.0+cu128 2.8.0.post1 | GPU: Tesla T4
Output: /kaggle/working/rq1_matched_metaifd_A_B_20260925T101044347775Z


## Hàm dùng chung: batches + ICVAE views dùng chung cho hai biến thể; `+` chỉ nối thêm 12 UFE


In [2]:
import torch.nn.functional as F
from scipy.stats import binomtest, t as student_t, wilcoxon
from sklearn.metrics import (average_precision_score, confusion_matrix, f1_score,
                             matthews_corrcoef, precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import train_test_split

EMPTY_CONTRASTIVE_NODE_TYPES = {"CA": 0, "EOA": 0}


def safe_contrast_module(self, ca_hidden_ls, eoa_hidden_ls):
    """Same patch as the audited Meta-IFD notebook: a node type absent from a batch contributes 0, not NaN."""
    def one_type(hidden_ls, node_type):
        if not hidden_ls or hidden_ls[0].shape[1] == 0:
            EMPTY_CONTRASTIVE_NODE_TYPES[node_type] += 1
            return None
        term = self.loss_co(self.attention([z[1] for z in hidden_ls], node_type),
                            self.attention([z[2] for z in hidden_ls], node_type),
                            self.attention([z[0] for z in hidden_ls], node_type))
        if not torch.isfinite(term):
            raise RuntimeError(f"Non-finite {node_type} contrastive term")
        return term
    terms = [v for v in (one_type(ca_hidden_ls, "CA"), one_type(eoa_hidden_ls, "EOA")) if v is not None]
    if not terms:
        return torch.zeros((), device=next(self.parameters()).device)
    return torch.stack(terms).sum()


HMSL.contrast_module = safe_contrast_module


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def compute_ufe_by_node_type(sampled_batch):
    """12 label-free statistics per node inside the sampled batch (identical to the audited Meta-IFD+)."""
    result = {nt: sampled_batch[nt].x.new_zeros((sampled_batch[nt].num_nodes, EDGE_STATS_DIM)) for nt in ("EOA", "CA")}
    for edge_type in sampled_batch.edge_types:
        source_type, relation, destination_type = edge_type
        store = sampled_batch[edge_type]
        if store.edge_index.numel() == 0:
            continue
        assert hasattr(store, "edge_attr") and store.edge_attr.shape[1] >= 2
        attributes = torch.nan_to_num(store.edge_attr[:, :2].to(dtype=sampled_batch[source_type].x.dtype),
                                      nan=0.0, posinf=0.0, neginf=0.0).clamp_min_(0.0)
        src, dst = store.edge_index[0], store.edge_index[1]
        ones = torch.ones(src.numel(), dtype=attributes.dtype, device=attributes.device)
        column = RELATION_TO_INDEX[relation] * 6
        result[destination_type][:, column + 0].index_add_(0, dst, ones)
        result[source_type][:, column + 1].index_add_(0, src, ones)
        result[destination_type][:, column + 2].index_add_(0, dst, attributes[:, 0])
        result[source_type][:, column + 3].index_add_(0, src, attributes[:, 0])
        result[destination_type][:, column + 4].index_add_(0, dst, attributes[:, 1])
        result[source_type][:, column + 5].index_add_(0, src, attributes[:, 1])
    result = {nt: torch.log1p(v) for nt, v in result.items()}
    assert all(bool(torch.isfinite(v).all()) for v in result.values())
    return result


@torch.no_grad()
def make_shared_views(sampled_batch):
    """One ICVAE realization shared by both variants: base views are 14-D; UFE is stored separately."""
    base_views = {"EOA": [], "CA": []}
    for _ in range(CONFIG["views"]):
        generated = {"EOA": [], "CA": []}
        for edge_type in all_edge_types:
            node_type = edge_type[0]
            num_nodes = sampled_batch[node_type].num_nodes
            edge_onehot = torch.zeros((num_nodes, len(all_edge_types)), device=device, dtype=torch.float32)
            edge_onehot[:, edge_type_to_id[edge_type]] = 1.0
            z = torch.randn((num_nodes, icvae_model.latent_size), device=device)
            augmented = icvae_model.inference(z, sampled_batch[node_type].x, edge_onehot)
            denominator = augmented.sum(dim=1, keepdim=True)
            augmented = torch.where(denominator != 0, augmented / denominator, torch.zeros_like(augmented))
            generated[node_type].append(augmented)
        for node_type in ("EOA", "CA"):
            base_views[node_type].append(torch.stack(generated[node_type] + [sampled_batch[node_type].x]).cpu())
    ufe = {nt: v.cpu() for nt, v in compute_ufe_by_node_type(sampled_batch).items()}
    return base_views, ufe


def fingerprint(items):
    digest = hashlib.sha256()
    for batch, base_views, ufe in items:
        for nt in ("EOA", "CA"):
            digest.update(batch[nt].n_id.detach().cpu().numpy().astype(np.int64, copy=False).tobytes())
            for view in base_views[nt]:
                digest.update(view.detach().cpu().numpy().astype(np.float32, copy=False).tobytes())
            digest.update(ufe[nt].detach().cpu().numpy().astype(np.float32, copy=False).tobytes())
        for et in sorted(batch.edge_types):
            digest.update(batch[et].edge_index.detach().cpu().numpy().astype(np.int64, copy=False).tobytes())
    return digest.hexdigest()


def materialize_split(mask, sampler_seed, view_seed, expected, shuffle):
    random.seed(sampler_seed); np.random.seed(sampler_seed)
    torch.manual_seed(sampler_seed); torch.cuda.manual_seed_all(sampler_seed)
    loader = NeighborLoader(data, input_nodes=("EOA", mask), num_neighbors=fanout_by_relation,
                            batch_size=CONFIG["batch_size"], shuffle=shuffle, num_workers=0)
    batches = [batch.cpu() for batch in loader]
    torch.manual_seed(view_seed); torch.cuda.manual_seed_all(view_seed)
    items, ids = [], []
    for batch in batches:
        n = int(batch["EOA"].batch_size)
        assert bool(torch.all((batch["EOA"].y[:n] == 0) | (batch["EOA"].y[:n] == 1)))
        ids.extend(batch["EOA"].n_id[:n].tolist())
        batch["EOA"].y[n:] = -1
        if "CA" in batch.node_types and hasattr(batch["CA"], "y"):
            batch["CA"].y[:] = -1
        gpu_batch = batch.to(device)
        base_views, ufe = make_shared_views(gpu_batch)
        batch = batch.cpu()
        items.append((batch, base_views, ufe))
        del gpu_batch
        torch.cuda.empty_cache()
    assert len(ids) == len(set(ids)) == expected
    assert set(ids) == set(torch.where(mask)[0].tolist())
    return items, ids


def variant_inputs(variant, base_views, ufe):
    views = {}
    for nt in ("EOA", "CA"):
        views[nt] = []
        for view in base_views[nt]:
            view = view.to(device)
            if variant == "plus":
                view = torch.cat([view, ufe[nt].to(device).unsqueeze(0).expand(view.shape[0], -1, -1)], dim=-1)
                assert view.shape[-1] == 26
            else:
                assert view.shape[-1] == 14
            views[nt].append(view)
    return views


def metrics(labels, scores, loss=None, seconds=None):
    labels = np.asarray(labels, dtype=np.int64)
    scores = np.asarray(scores, dtype=np.float64)
    predictions = (scores >= CONFIG["threshold"]).astype(np.int64)
    tn, fp, fn, tp = confusion_matrix(labels, predictions, labels=[0, 1]).ravel()
    result = {
        "pr_auc": float(average_precision_score(labels, scores)), "roc_auc": float(roc_auc_score(labels, scores)),
        "precision": float(precision_score(labels, predictions, zero_division=0)),
        "recall": float(recall_score(labels, predictions, zero_division=0)),
        "binary_f1": float(f1_score(labels, predictions, zero_division=0)),
        "micro_f1": float(f1_score(labels, predictions, average="micro")),
        "macro_f1": float(f1_score(labels, predictions, average="macro")),
        "mcc": float(matthews_corrcoef(labels, predictions)), "fpr": float(fp / (fp + tn)),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }
    if loss is not None:
        result["loss"] = float(loss)
    if seconds is not None:
        result["seconds"] = float(seconds)
    return result


def run_items(model, variant, items, optimizer=None, collect_rows=False):
    training = optimizer is not None
    model.train(training)
    labels_all, scores_all, losses, rows = [], [], [], []
    if device.type == "cuda":
        torch.cuda.synchronize()
    started = time.perf_counter()
    for cpu_batch, base_views, ufe in items:
        batch = cpu_batch.to(device)
        views = variant_inputs(variant, base_views, ufe)
        n = int(batch["EOA"].batch_size)
        labels = batch["EOA"].y[:n]
        try:
            if training:
                optimizer.zero_grad(set_to_none=True)
            with torch.set_grad_enabled(training):
                logits, contrastive = model(views, batch.edge_index_dict)
                loss = F.cross_entropy(logits[:n], labels) + CONFIG["contrastive_lambda"] * contrastive
                if not torch.isfinite(loss):
                    raise RuntimeError("Non-finite Meta-IFD loss")
                if training:
                    loss.backward()
                    optimizer.step()
            probabilities = torch.softmax(logits[:n].detach(), dim=1)[:, 1]
            label_values = labels.detach().cpu().tolist()
            score_values = probabilities.detach().cpu().tolist()
            labels_all.extend(label_values)
            scores_all.extend(score_values)
            losses.append(float(loss.detach().cpu()) * n)
            if collect_rows:
                ids = batch["EOA"].n_id[:n].detach().cpu().tolist()
                rows.extend({"node_id": int(i), "label": int(l), "score": float(s),
                             "prediction": int(s >= CONFIG["threshold"]), "threshold": CONFIG["threshold"]}
                            for i, l, s in zip(ids, label_values, score_values))
        finally:
            del batch, views
    if device.type == "cuda":
        torch.cuda.synchronize()
    return metrics(labels_all, scores_all, loss=sum(losses) / len(labels_all),
                   seconds=time.perf_counter() - started), rows


def build_model(variant, seed, template_items):
    """Materialise lazy Linear(-1, .) layers, then re-initialise every parameter from `seed`."""
    set_seed(seed)
    model = HMSL(hidden=CONFIG["hidden"], out_channels=2, data=template_items[0][0],
                 concat=CONFIG["views"], target_node=CONFIG["target_node"]).to(device)
    batch = template_items[0][0].to(device)
    with torch.no_grad():
        model.eval()
        model(variant_inputs(variant, template_items[0][1], template_items[0][2]), batch.edge_index_dict)
    del batch
    set_seed(seed)
    model.reset_parameters()
    return model


def clone_state(model):
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


def write_csv(path, rows):
    assert rows
    with Path(path).open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def train_one(variant, seed, train_items, validation_items, folder):
    model = build_model(variant, seed, train_items)
    parameters = sum(p.numel() for p in model.parameters())
    optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["learning_rate"], weight_decay=CONFIG["weight_decay"])
    best_macro, best_epoch, best_state, stale, history = -1.0, 0, None, 0, []
    started = time.perf_counter()
    for epoch in range(1, CONFIG["max_epochs"] + 1):
        train_result, _ = run_items(model, variant, train_items, optimizer=optimizer)
        val_result, _ = run_items(model, variant, validation_items)
        if val_result["macro_f1"] >= best_macro:  # official Meta-IFD rule
            best_macro, best_epoch, best_state, stale = val_result["macro_f1"], epoch, clone_state(model), 0
        else:
            stale += 1
        history.append({"epoch": epoch, "train_loss": train_result["loss"], "train_macro_f1": train_result["macro_f1"],
                        "validation_loss": val_result["loss"], "validation_macro_f1": val_result["macro_f1"],
                        "validation_pr_auc": val_result["pr_auc"], "validation_fpr": val_result["fpr"],
                        "best_epoch": best_epoch, "stale_epochs": stale})
        if stale > CONFIG["patience"]:
            break
    training_seconds = time.perf_counter() - started
    model.load_state_dict(best_state, strict=True)
    val_final, _ = run_items(model, variant, validation_items)
    assert abs(val_final["macro_f1"] - best_macro) < 5e-3, (val_final["macro_f1"], best_macro)
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / f"{variant}_seed_{seed}.pt"
    torch.save({"state_dict": best_state, "variant": variant, "seed": seed, "best_epoch": best_epoch,
                "trained_epochs": len(history), "parameter_count": parameters,
                "protocol_sha256": PROTOCOL_SHA256, "source_data_sha256": data_sha256}, path)
    write_csv(folder / f"history_{variant}_seed_{seed}.csv", history)
    best_row = history[best_epoch - 1]
    row = {"variant": variant, "seed": seed, "best_epoch": best_epoch, "trained_epochs": len(history),
           "training_seconds": training_seconds, "parameter_count": parameters,
           "train_macro_f1_at_best": best_row["train_macro_f1"],
           **{f"validation_{k}": v for k, v in val_final.items()}, "checkpoint_sha256": sha256_file(path)}
    del model, optimizer, best_state
    gc.collect()
    torch.cuda.empty_cache()
    return row, path


def evaluate_checkpoint(variant, seed, path, expected_hash, test_items, template_items):
    assert sha256_file(path) == expected_hash
    checkpoint = torch.load(path, map_location="cpu", weights_only=False)
    assert checkpoint["variant"] == variant and checkpoint["seed"] == seed
    assert checkpoint["protocol_sha256"] == PROTOCOL_SHA256
    model = build_model(variant, seed, template_items)
    model.load_state_dict(checkpoint["state_dict"], strict=True)
    assert sum(p.numel() for p in model.parameters()) == checkpoint["parameter_count"]
    result, rows = run_items(model, variant, test_items, collect_rows=True)
    del model, checkpoint
    gc.collect()
    torch.cuda.empty_cache()
    return result, rows


def mean_macro(score_matrix, labels, sample):
    truth = labels[sample].astype(bool)
    pred = score_matrix[:, sample] >= CONFIG["threshold"]
    tp = (pred & truth).sum(1); fp = (pred & ~truth).sum(1)
    fn = (~pred & truth).sum(1); tn = (~pred & ~truth).sum(1)
    f1_pos = np.divide(2 * tp, 2 * tp + fp + fn, out=np.zeros(len(tp)), where=(2 * tp + fp + fn) > 0)
    f1_neg = np.divide(2 * tn, 2 * tn + fp + fn, out=np.zeros(len(tn)), where=(2 * tn + fp + fn) > 0)
    return float(np.mean((f1_pos + f1_neg) / 2))


def run_matched_split(split_key, masks, sampler_seeds, view_seeds):
    """Train both variants x 5 seeds, freeze, then (only then) sample test and evaluate."""
    folder = run_dir / split_key
    folder.mkdir(parents=True, exist_ok=False)
    train_items, train_ids = materialize_split(masks["train"], sampler_seeds["train"], view_seeds["train"], EXPECTED_COUNTS["train"], True)
    validation_items, validation_ids = materialize_split(masks["validation"], sampler_seeds["validation"], view_seeds["validation"], EXPECTED_COUNTS["validation"], True)
    assert set(train_ids).isdisjoint(validation_ids)
    shared = {"train_items_sha256": fingerprint(train_items), "validation_items_sha256": fingerprint(validation_items)}
    rows, paths = [], {}
    for variant in VARIANTS:
        for seed in MODEL_SEEDS:
            row, path = train_one(variant, seed, train_items, validation_items, folder)
            rows.append(row); paths[(variant, seed)] = path
            print(f"[{split_key}] {VARIANT_LABEL[variant]} seed={seed} val Macro-F1={row['validation_macro_f1']:.4f} "
                  f"epoch={row['best_epoch']}/{row['trained_epochs']}", flush=True)
    write_csv(folder / "validation_runs.csv", rows)
    decision = {"status": "FROZEN VALIDATION DECISION (both variants)", "split_key": split_key,
                "protocol_sha256": PROTOCOL_SHA256, "source_data_sha256": data_sha256,
                "sampler_seeds": sampler_seeds, "view_seeds": view_seeds,
                "train_target_ids_sha256": sha256_int64(sorted(train_ids)),
                "validation_target_ids_sha256": sha256_int64(sorted(validation_ids)), **shared,
                "checkpoint_sha256": {f"{r['variant']}_seed_{r['seed']}": r["checkpoint_sha256"] for r in rows},
                "test_batches_sampled_before_freeze": False, "test_metrics_computed_before_freeze": False}
    (folder / "frozen_validation_decision.json").write_text(json.dumps(decision, indent=2), encoding="utf-8")

    test_items, test_ids = materialize_split(masks["test"], sampler_seeds["test"], view_seeds["test"], EXPECTED_COUNTS["test"], False)
    assert set(test_ids).isdisjoint(train_ids) and set(test_ids).isdisjoint(validation_ids)
    results, predictions, cube = [], [], {}
    for variant in VARIANTS:
        scores = []
        for seed in MODEL_SEEDS:
            result, prow = evaluate_checkpoint(variant, seed, paths[(variant, seed)],
                                               decision["checkpoint_sha256"][f"{variant}_seed_{seed}"], test_items, train_items)
            assert [r["node_id"] for r in prow] == test_ids
            scores.append([r["score"] for r in prow])
            for r in prow:
                r.update({"split_key": split_key, "variant": variant, "model": VARIANT_LABEL[variant], "seed": seed})
            predictions.extend(prow)
            vrow = next(r for r in rows if r["variant"] == variant and r["seed"] == seed)
            results.append({"split_key": split_key, "variant": variant, "model": VARIANT_LABEL[variant], "seed": seed,
                            "best_epoch": vrow["best_epoch"], "validation_macro_f1": vrow["validation_macro_f1"],
                            **{f"test_{k}": v for k, v in result.items()}})
        cube[variant] = np.asarray(scores)
    labels = np.asarray([r["label"] for r in predictions if r["variant"] == "base" and r["seed"] == MODEL_SEEDS[0]])
    write_csv(folder / "test_results.csv", results)
    write_csv(folder / "test_predictions.csv", predictions)
    split_result = {
        "split_key": split_key, "protocol_sha256": PROTOCOL_SHA256,
        "test_target_ids_sha256": sha256_int64(test_ids), "test_items_sha256": fingerprint(test_items),
        "base_mean_macro_f1": float(np.mean([r["test_macro_f1"] for r in results if r["variant"] == "base"])),
        "plus_mean_macro_f1": float(np.mean([r["test_macro_f1"] for r in results if r["variant"] == "plus"])),
        "base_mean_validation_macro_f1": float(np.mean([r["validation_macro_f1"] for r in results if r["variant"] == "base"])),
        "plus_mean_validation_macro_f1": float(np.mean([r["validation_macro_f1"] for r in results if r["variant"] == "plus"])),
        "decision": decision, "code_sha256": CODE_SHA256, "source_sha256": EXPECTED_SOURCE_SHA256,
        "source_data_sha256": data_sha256,
        "test_target_ids_sorted_sha256": sha256_int64(sorted(test_ids)),
    }
    split_result["delta"] = split_result["plus_mean_macro_f1"] - split_result["base_mean_macro_f1"]
    np.savez_compressed(folder / "score_cube.npz", base=cube["base"], plus=cube["plus"], labels=labels,
                        test_ids=np.asarray(test_ids))
    split_result["score_cube_sha256"] = sha256_file(folder / "score_cube.npz")
    (folder / "split_result.json").write_text(json.dumps(split_result, indent=2), encoding="utf-8")
    # All checkpoints are kept (hashes in frozen_validation_decision.json) so they can be re-verified later.
    del train_items, validation_items, test_items
    gc.collect()
    torch.cuda.empty_cache()
    return split_result, cube, labels


def verify_previous_split(split_key, item, source, masks, sampler_seeds, view_seeds):
    """Return the reloaded cube only if every integrity check passes; otherwise None (split is re-run)."""
    problems = []
    decision = item.get("decision", {})
    checks = {
        "protocol_sha256": (item.get("protocol_sha256"), PROTOCOL_SHA256),
        "code_sha256": (item.get("code_sha256"), CODE_SHA256),
        "source_data_sha256": (item.get("source_data_sha256"), data_sha256),
        "source_sha256": (item.get("source_sha256"), EXPECTED_SOURCE_SHA256),
        "sampler_seeds": (decision.get("sampler_seeds"), sampler_seeds),
        "view_seeds": (decision.get("view_seeds"), view_seeds),
        "train_ids": (decision.get("train_target_ids_sha256"), sha256_int64(sorted(torch.where(masks["train"])[0].tolist()))),
        "validation_ids": (decision.get("validation_target_ids_sha256"), sha256_int64(sorted(torch.where(masks["validation"])[0].tolist()))),
        "test_ids": (item.get("test_target_ids_sorted_sha256"), sha256_int64(sorted(torch.where(masks["test"])[0].tolist()))),
    }
    for name, (got, want) in checks.items():
        if got != want:
            problems.append(name)
    cube_path = source / "score_cube.npz"
    if not cube_path.is_file() or sha256_file(cube_path) != item.get("score_cube_sha256"):
        problems.append("score_cube_sha256")
    for key, digest in decision.get("checkpoint_sha256", {}).items():
        ckpt = source / f"{key}.pt"
        if not ckpt.is_file() or sha256_file(ckpt) != digest:
            problems.append(f"checkpoint:{key}")
    if not problems:
        npz = np.load(cube_path)
        nodes = np.arange(len(npz["labels"]))
        for variant, field in (("base", "base_mean_macro_f1"), ("plus", "plus_mean_macro_f1")):
            if abs(mean_macro(npz[variant], npz["labels"], nodes) - item[field]) > 1e-9:
                problems.append(f"recomputed_{variant}_macro_f1")
        if sha256_int64(sorted(npz["test_ids"].tolist())) != item.get("test_target_ids_sorted_sha256"):
            problems.append("cube_test_ids")
    if problems:
        print(f"[{split_key}] previous result REJECTED ({', '.join(problems)}) -> re-running", flush=True)
        return None
    return {"base": npz["base"], "plus": npz["plus"]}, npz["labels"]


def reuse_or_run(split_key, masks, sampler_seeds, view_seeds):
    if split_key in previous_split_results:
        item, source = previous_split_results[split_key]
        verified = verify_previous_split(split_key, item, source, masks, sampler_seeds, view_seeds)
        if verified is not None:
            shutil.copytree(source, run_dir / split_key)
            print(f"[{split_key}] reused from previous session after full integrity check", flush=True)
            return item, verified[0], verified[1]
    return run_matched_split(split_key, masks, sampler_seeds, view_seeds)


print("Functions ready. Protocol:", PROTOCOL_SHA256)


Functions ready. Protocol: 29a9e98c57a2984aea0d84a2b3671b1da1c3a0a9783ee5082891820744b9262e


## Bảng A — split paper (retrospective). Test chỉ được lấy mẫu sau khi khóa checkpoint cả hai biến thể


In [3]:
# ===== Bảng A: split paper (retrospective matched comparison) =====
result_a, cube_a, labels_a = reuse_or_run("A_paper_split", paper_masks, PAPER_SAMPLER_SEEDS, PAPER_VIEW_SEEDS)
all_nodes = np.arange(len(labels_a))
observed_a = mean_macro(cube_a["plus"], labels_a, all_nodes) - mean_macro(cube_a["base"], labels_a, all_nodes)
rng = np.random.default_rng(20260925)
boot = np.empty(BOOTSTRAP_REPETITIONS)
for i in range(BOOTSTRAP_REPETITIONS):
    sample = rng.integers(0, len(labels_a), size=len(labels_a))
    boot[i] = mean_macro(cube_a["plus"], labels_a, sample) - mean_macro(cube_a["base"], labels_a, sample)
ci_a = np.quantile(boot, [0.025, 0.975])
p_a = min(1.0, 2 * min((np.sum(boot <= 0) + 1) / (BOOTSTRAP_REPETITIONS + 1),
                       (np.sum(boot >= 0) + 1) / (BOOTSTRAP_REPETITIONS + 1)))
per_seed_a = []
for k, seed in enumerate(MODEL_SEEDS):
    plus_ok = (cube_a["plus"][k] >= 0.5) == labels_a
    base_ok = (cube_a["base"][k] >= 0.5) == labels_a
    only_plus, only_base = int(np.sum(plus_ok & ~base_ok)), int(np.sum(~plus_ok & base_ok))
    per_seed_a.append({"seed": seed,
                       "plus_macro_f1": mean_macro(cube_a["plus"][[k]], labels_a, all_nodes),
                       "base_macro_f1": mean_macro(cube_a["base"][[k]], labels_a, all_nodes),
                       "only_plus_correct": only_plus, "only_base_correct": only_base,
                       "mcnemar_p": 1.0 if only_plus + only_base == 0 else float(binomtest(min(only_plus, only_base), only_plus + only_base, 0.5).pvalue)})
table_a = {"role": "retrospective matched comparison on the previously viewed paper split",
           "base_mean_macro_f1": result_a["base_mean_macro_f1"], "plus_mean_macro_f1": result_a["plus_mean_macro_f1"],
           "delta": observed_a, "bootstrap_95ci": [float(ci_a[0]), float(ci_a[1])], "bootstrap_p_two_sided": float(p_a),
           "seeds_plus_better": int(sum(r["plus_macro_f1"] > r["base_macro_f1"] for r in per_seed_a)), "per_seed": per_seed_a}
(run_dir / "table_a_summary.json").write_text(json.dumps(table_a, indent=2), encoding="utf-8")
print(f"BẢNG A — Meta-IFD {result_a['base_mean_macro_f1']*100:.2f} | Meta-IFD+ {result_a['plus_mean_macro_f1']*100:.2f} | "
      f"Δ {observed_a*100:+.2f} pp, 95% CI [{ci_a[0]*100:+.2f}, {ci_a[1]*100:+.2f}], p={p_a:.4f}, "
      f"seeds plus better {table_a['seeds_plus_better']}/5")


[A_paper_split] Meta-IFD seed=12 val Macro-F1=0.9890 epoch=167/218
[A_paper_split] Meta-IFD seed=13 val Macro-F1=0.9835 epoch=169/220
[A_paper_split] Meta-IFD seed=14 val Macro-F1=0.9816 epoch=25/76
[A_paper_split] Meta-IFD seed=15 val Macro-F1=0.9834 epoch=78/129
[A_paper_split] Meta-IFD seed=16 val Macro-F1=0.9816 epoch=15/66
[A_paper_split] Meta-IFD+ seed=12 val Macro-F1=0.9927 epoch=54/105
[A_paper_split] Meta-IFD+ seed=13 val Macro-F1=0.9908 epoch=48/99
[A_paper_split] Meta-IFD+ seed=14 val Macro-F1=0.9890 epoch=38/89
[A_paper_split] Meta-IFD+ seed=15 val Macro-F1=0.9908 epoch=27/78
[A_paper_split] Meta-IFD+ seed=16 val Macro-F1=0.9890 epoch=33/84
BẢNG A — Meta-IFD 97.65 | Meta-IFD+ 98.60 | Δ +0.96 pp, 95% CI [+0.18, +1.81], p=0.0139, seeds plus better 5/5


## Bảng B — 10 split phân tầng mới (paper split không nằm trong đây)


In [4]:
# ===== Bảng B: 10 split phân tầng mới đã khóa trước =====
split_results_b = []
for split_seed in RESPLIT_SEEDS:
    positions = np.arange(len(labeled_ids))
    trainval_pos, test_pos = train_test_split(positions, test_size=0.20, random_state=split_seed, stratify=labeled_y)
    train_pos, validation_pos = train_test_split(trainval_pos, test_size=0.25, random_state=split_seed,
                                                 stratify=labeled_y[trainval_pos])
    assert (len(train_pos), len(validation_pos), len(test_pos)) == (1657, 553, 553)
    masks = {}
    for name, pos in (("train", train_pos), ("validation", validation_pos), ("test", test_pos)):
        m = torch.zeros(data["EOA"].num_nodes, dtype=torch.bool)
        m[torch.from_numpy(labeled_ids[pos])] = True
        masks[name] = m
    assert not bool((masks["train"] & masks["validation"]).any() or (masks["train"] & masks["test"]).any()
                    or (masks["validation"] & masks["test"]).any())
    sampler_seeds = {"train": split_seed * 10 + 1, "validation": split_seed * 10 + 2, "test": split_seed * 10 + 3}
    view_seeds = {"train": split_seed * 10 + 4, "validation": split_seed * 10 + 5, "test": split_seed * 10 + 6}
    result, _, _ = reuse_or_run(f"B_split_{split_seed}", masks, sampler_seeds, view_seeds)
    split_results_b.append(result)
    print(f"[B_split_{split_seed}] Meta-IFD {result['base_mean_macro_f1']*100:.2f} | Meta-IFD+ {result['plus_mean_macro_f1']*100:.2f} | "
          f"Δ {result['delta']*100:+.2f} pp", flush=True)


[B_split_101] Meta-IFD seed=12 val Macro-F1=0.9705 epoch=25/76
[B_split_101] Meta-IFD seed=13 val Macro-F1=0.9761 epoch=109/160
[B_split_101] Meta-IFD seed=14 val Macro-F1=0.9706 epoch=46/97
[B_split_101] Meta-IFD seed=15 val Macro-F1=0.9743 epoch=45/96
[B_split_101] Meta-IFD seed=16 val Macro-F1=0.9761 epoch=50/101
[B_split_101] Meta-IFD+ seed=12 val Macro-F1=0.9871 epoch=65/116
[B_split_101] Meta-IFD+ seed=13 val Macro-F1=0.9835 epoch=30/81
[B_split_101] Meta-IFD+ seed=14 val Macro-F1=0.9835 epoch=51/102
[B_split_101] Meta-IFD+ seed=15 val Macro-F1=0.9816 epoch=58/109
[B_split_101] Meta-IFD+ seed=16 val Macro-F1=0.9853 epoch=71/122
[B_split_101] Meta-IFD 97.02 | Meta-IFD+ 97.65 | Δ +0.63 pp
[B_split_202] Meta-IFD seed=12 val Macro-F1=0.9725 epoch=37/88
[B_split_202] Meta-IFD seed=13 val Macro-F1=0.9707 epoch=31/82
[B_split_202] Meta-IFD seed=14 val Macro-F1=0.9707 epoch=56/107
[B_split_202] Meta-IFD seed=15 val Macro-F1=0.9743 epoch=23/74
[B_split_202] Meta-IFD seed=16 val Macro-F1=0

## Kiểm định chính: Nadeau–Bengio trên 10 chênh lệch cấp split


In [5]:
# ===== Kiểm định chính: Nadeau–Bengio trên 10 chênh lệch cấp split =====
deltas = np.asarray([r["delta"] for r in split_results_b])
k = len(deltas)
assert k == len(RESPLIT_SEEDS) == 10
ratio = EXPECTED_COUNTS["test"] / EXPECTED_COUNTS["train"]
variance = float(deltas.var(ddof=1))
se = float(np.sqrt((1 / k + ratio) * variance))
mean_delta = float(deltas.mean())
if se == 0:
    t_stat, p_nb, ci_nb = (float("inf") if mean_delta else 0.0), (0.0 if mean_delta else 1.0), (mean_delta, mean_delta)
else:
    t_stat = mean_delta / se
    p_nb = float(2 * student_t.sf(abs(t_stat), df=k - 1))
    crit = float(student_t.ppf(0.975, df=k - 1))
    ci_nb = (mean_delta - crit * se, mean_delta + crit * se)
try:
    wilcoxon_p = float(wilcoxon(deltas).pvalue)
except ValueError:
    wilcoxon_p = 1.0
improves = bool(mean_delta >= MINIMUM_PRACTICAL_DELTA and ci_nb[0] > 0)
table_b = {
    "role": "pre-registered re-split stability; not an external holdout; paper split excluded",
    "split_seeds": RESPLIT_SEEDS,
    "per_split": [{"split_key": r["split_key"], "base": r["base_mean_macro_f1"], "plus": r["plus_mean_macro_f1"],
                   "delta": r["delta"], "base_val": r["base_mean_validation_macro_f1"],
                   "plus_val": r["plus_mean_validation_macro_f1"]} for r in split_results_b],
    "base_mean_over_splits": float(np.mean([r["base_mean_macro_f1"] for r in split_results_b])),
    "plus_mean_over_splits": float(np.mean([r["plus_mean_macro_f1"] for r in split_results_b])),
    "mean_delta": mean_delta, "sd_delta": float(np.sqrt(variance)),
    "nadeau_bengio": {"se": se, "t": t_stat, "df": k - 1, "p_two_sided": p_nb, "ci95": [ci_nb[0], ci_nb[1]],
                      "ratio_test_train": ratio},
    "descriptive_wilcoxon_p": wilcoxon_p,
    "splits_plus_better": int(np.sum(deltas > 0)), "splits_tied": int(np.sum(deltas == 0)),
    "decision_rule": CONFIG["decision_rule"], "ufe_improves_metaifd": improves,
}
write_csv(run_dir / "table_b_per_split.csv", table_b["per_split"])
final_summary = {"status": "completed matched Meta-IFD vs Meta-IFD+ (Table A + Table B)",
                 "protocol_sha256": PROTOCOL_SHA256, "protocol": CONFIG, "source_data_sha256": data_sha256,
                 "source_sha256": EXPECTED_SOURCE_SHA256, "table_a": table_a, "table_b": table_b,
                 "empty_contrastive_node_types": EMPTY_CONTRASTIVE_NODE_TYPES,
                 "post_test_selection": False, "protocol_changed_after_results": False}
(run_dir / "final_summary.json").write_text(json.dumps(final_summary, indent=2), encoding="utf-8")
shutil.make_archive(str(run_dir), "zip", run_dir)

print("\nBẢNG A (split paper, retrospective)")
print(f"  Meta-IFD {table_a['base_mean_macro_f1']*100:.2f} | Meta-IFD+ {table_a['plus_mean_macro_f1']*100:.2f} | "
      f"Δ {table_a['delta']*100:+.2f} pp, CI [{table_a['bootstrap_95ci'][0]*100:+.2f}, {table_a['bootstrap_95ci'][1]*100:+.2f}], p={table_a['bootstrap_p_two_sided']:.4f}")
print("BẢNG B (10 split mới, pre-registered)")
for r in table_b["per_split"]:
    print(f"  {r['split_key']}: {r['base']*100:.2f} → {r['plus']*100:.2f} (Δ {r['delta']*100:+.2f})")
print(f"  Mean Δ {mean_delta*100:+.2f} pp | NB 95% CI [{ci_nb[0]*100:+.2f}, {ci_nb[1]*100:+.2f}] | p={p_nb:.4f} | "
      f"splits plus better {table_b['splits_plus_better']}/10")
print("UFE improves Meta-IFD (pre-declared rule, Table B):", improves)
print("Output:", run_dir, "| ZIP:", str(run_dir) + ".zip")



BẢNG A (split paper, retrospective)
  Meta-IFD 97.65 | Meta-IFD+ 98.60 | Δ +0.96 pp, CI [+0.18, +1.81], p=0.0139
BẢNG B (10 split mới, pre-registered)
  B_split_101: 97.02 → 97.65 (Δ +0.63)
  B_split_202: 98.20 → 99.34 (Δ +1.13)
  B_split_303: 98.31 → 98.49 (Δ +0.18)
  B_split_404: 96.50 → 97.90 (Δ +1.40)
  B_split_505: 97.07 → 97.40 (Δ +0.33)
  B_split_606: 97.07 → 98.68 (Δ +1.61)
  B_split_707: 96.13 → 97.61 (Δ +1.49)
  B_split_808: 97.39 → 97.36 (Δ -0.03)
  B_split_909: 97.13 → 98.16 (Δ +1.03)
  B_split_1010: 98.31 → 99.08 (Δ +0.77)
  Mean Δ +0.85 pp | NB 95% CI [-0.00, +1.71] | p=0.0500 | splits plus better 9/10
UFE improves Meta-IFD (pre-declared rule, Table B): False
Output: /kaggle/working/rq1_matched_metaifd_A_B_20260925T101044347775Z | ZIP: /kaggle/working/rq1_matched_metaifd_A_B_20260925T101044347775Z.zip
